### Import Libraries

In [1]:
import duckdb
import pandas as pd
import numpy as np
import unicodedata
import re

### DuckDB Connection

In [2]:
conn = duckdb.connect('/app/data/analytics.duckdb')

### Source Data

#### All Reviews from Staging Reviews model

In [3]:
## All reviews
raw_reviews = conn.execute("""
                    SELECT distinct 
                        review_id, review_score, review_comment_message, review_comment_title
                    FROM staging.stg_order_reviews
                """).df()

raw_reviews['message_length'] = raw_reviews['review_comment_message'].str.len()
print("Shape -", raw_reviews.shape)
raw_reviews.head(n=2)

Shape - (98410, 5)


,review_id,review_score,review_comment_message,review_comment_title,message_length
0,72730619a00e0ecdecf0a4ff862d8996,1,comprei tres pacotes de cinco folhas cada de p...,None,95.0
1,56c957e848b3f71bcebd87e12b7883d7,4,None,None,NaN


#### Source Portugese to English Translations for short review messages/titles

In [4]:
lookup_df = pd.read_csv('/app/short_review_lookup.csv')
short_review_lookup = dict(zip(lookup_df['portuguese'], lookup_df['english']))

#### Translate Short Messages/Titles

In [5]:
def normalize(text):
    text = text.lower().strip()
    # Remove accents
    text = unicodedata.normalize('NFKD', text)
    text = ''.join(c for c in text if not unicodedata.combining(c))
    # Remove special chars
    text = re.sub(r'[^a-z ]', '', text)
    # Collapse spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [6]:
# Apply lookup only for short messages
short_reviews = raw_reviews['review_comment_message'].notna() & (raw_reviews['review_comment_message'].str.len() < 10)

raw_reviews.loc[short_reviews, 'normalized_message'] = raw_reviews.loc[short_reviews, 'review_comment_message'].apply(normalize)
raw_reviews.loc[short_reviews, 'translated_message'] = raw_reviews.loc[short_reviews, 'normalized_message'].map(short_review_lookup)

short_titles = raw_reviews['review_comment_title'].notna() & (raw_reviews['review_comment_title'].str.len() < 10)
raw_reviews.loc[short_titles, 'normalized_title'] = raw_reviews.loc[short_titles, 'review_comment_title'].apply(normalize)
raw_reviews.loc[short_titles, 'translated_title'] = raw_reviews.loc[short_titles, 'normalized_title'].map(short_review_lookup)


print(f"Short reviews mapped: {raw_reviews['translated_message'].notna().sum():,}")
print(f"Short reviews unmapped (noise): {short_reviews.sum() - raw_reviews['translated_message'].notna().sum():,}")
print(f"Short reviews mapped: {raw_reviews['translated_title'].notna().sum():,}")
print(f"Short reviews unmapped (noise): {short_titles.sum() - raw_reviews['translated_title'].notna().sum():,}")

raw_reviews.head(n=3)

Short reviews mapped: 2,491
Short reviews unmapped (noise): 442
Short reviews mapped: 3,916
Short reviews unmapped (noise): 1,113


,review_id,review_score,review_comment_message,review_comment_title,message_length,normalized_message,translated_message,normalized_title,translated_title
0,72730619a00e0ecdecf0a4ff862d8996,1,comprei tres pacotes de cinco folhas cada de p...,None,95.0,NaN,NaN,NaN,NaN
1,56c957e848b3f71bcebd87e12b7883d7,4,None,None,NaN,NaN,NaN,NaN,NaN
2,225f4f306d90cbc502d3c781d3015ed2,5,None,None,NaN,NaN,NaN,NaN,NaN


#### Load LLM Translations

In [7]:
llm_translations = conn.execute("""
    SELECT *
    FROM llm_outputs.translated_reviews
""").df()

print("Shape -", llm_translations.shape)
llm_translations.head(n=3)

Shape - (39946, 3)


,review_id,translated_message,translated_title
0,00020c7512a52e92212f12d3e37513c0,The delivery was super fast and the pendant is...,Fast delivery!
1,00046a69550325aea5fb89f65c7387f2,"I liked the phone case, it arrived as I expected.",I HIGHLY RECOMMEND
2,0005534973388c830bb858cfba83b17b,Excellent product. Deadline met. Flavor is als...,None


#### Join Raw and LLM Data

In [9]:
raw_reviews = raw_reviews.merge(llm_translations, on='review_id', how='left')
raw_reviews['translated_message'] = raw_reviews['translated_message_x'].combine_first(raw_reviews['translated_message_y'])
raw_reviews['translated_title'] = raw_reviews['translated_title_x'].combine_first(raw_reviews['translated_title_y'])
raw_reviews = raw_reviews.drop(columns=['translated_message_x', 'translated_message_y', 
                       'translated_title_x', 'translated_title_y',
                       'normalized_message', 'normalized_title'])
raw_reviews.head()

,review_id,review_score,review_comment_message,review_comment_title,message_length,translated_message,translated_title
0,72730619a00e0ecdecf0a4ff862d8996,1,comprei tres pacotes de cinco folhas cada de p...,None,95.0,I bought three packages of five sheets each of...,None
1,56c957e848b3f71bcebd87e12b7883d7,4,None,None,NaN,NaN,NaN
2,225f4f306d90cbc502d3c781d3015ed2,5,None,None,NaN,NaN,NaN
3,30a4f0f115950517b46d48046b609bd1,5,Tive um pós venda excelente através do Sr Renato,Excelente,48.0,I had excellent after-sales service through Mr...,excellent
4,3a3ae0e3f4a7d6b4d1cbdf1b1732e424,4,None,None,NaN,NaN,NaN


#### Dataset Overview

In [10]:
print(f"Total reviews: {len(raw_reviews):,}")
print(f"\nMessage coverage:")
print(f"  Has original message: {raw_reviews['review_comment_message'].notna().sum():,}")
print(f"  Has translated message: {raw_reviews['translated_message'].notna().sum():,}")
print(f"  No message at all: {raw_reviews['review_comment_message'].isna().sum():,}")

print(f"\nTitle coverage:")
print(f"  Has original title: {raw_reviews['review_comment_title'].notna().sum():,}")
print(f"  Has translated title: {raw_reviews['translated_title'].notna().sum():,}")

print(f"\nReview score distribution:")
print(raw_reviews['review_score'].value_counts().sort_index())

Total reviews: 98,410

Message coverage:
  Has original message: 40,668
  Has translated message: 40,209
  No message at all: 57,742

Title coverage:
  Has original title: 11,519
  Has translated title: 11,174

Review score distribution:
review_score
1    11282
2     3114
3     8097
4    19007
5    56910
Name: count, dtype: int64


In [11]:
conn.execute("CREATE SCHEMA IF NOT EXISTS llm_outputs")
conn.execute("DROP TABLE IF EXISTS llm_outputs.review_translations_final")
conn.execute("CREATE TABLE llm_outputs.review_translations_final AS SELECT * FROM raw_reviews")

print(f"Saved {len(raw_reviews):,} rows to llm_outputs.review_translations_final")

Saved 98,410 rows to llm_outputs.review_translations_final
